# 02 — Inverse QSAR on Tubulin (GA)

End-to-end recipe for de novo design of putative Tubulin inhibitors:

1. Load 603 known Tubulin actives/inactives encoded as VQGAE fragment counts,
   plus a pretrained Random Forest classifier.
2. Run a Genetic Algorithm over the 4096-dim fragment-count space, optimizing
   `0.5·RF + 0.3·dissimilarity − size_penalty`.
3. Decode the GA's solutions; filter by validity + structure rules.

Data fixture lives at `../data/tubulin/`. The encoding + RF training step
that produced these is documented at the end of the notebook.


## Imports

In [ ]:
import pickle
import numpy as np
import pandas as pd
import pygad
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from chython.files import SDFWrite

from VQGAE import (
    VQGAE, OrderingNetwork,
    decode_population, tanimoto_kernel, filter_molecule,
)

## Load fragment-count features + the QSAR classifier

In [ ]:
DATA = "../data/tubulin"
X = np.load(f"{DATA}/tubulin_qsar_class_train_data_vqgae.npz")["x"]   # (603, 4096)
Y = np.load(f"{DATA}/tubulin_qsar_class_train_data_vqgae.npz")["y"]   # (603,)
with open(f"{DATA}/rf_class_train_tubulin.pickle", "rb") as fh:
    rf_model = pickle.load(fh)

print("X:", X.shape, "Y:", Y.shape, "active fraction:", round(float(Y.mean()), 3))
print("RF balanced accuracy (10-fold CV):", round(rf_model.best_score_, 3))

## Load VQGAE + OrderingNetwork

In [ ]:
REPO = "tagirshin/VQGAE"
BATCH = 500
vqgae_ckpt = hf_hub_download(REPO, "vqgae.ckpt")
onn_ckpt   = hf_hub_download(REPO, "ordering_network.ckpt")
ordering_model = OrderingNetwork.load_from_checkpoint(onn_ckpt, batch_size=BATCH, map_location="cpu").eval()

## Fitness function

The GA operates over the 4096-dim integer fragment-count space.
Each candidate is scored by:

- `rf_score`     — RF probability of being active, ∈ [0, 1]
- `size_penalty` — −1 if heavy-atom count < 18, else 0 (avoid tiny hits)
- `dissimilarity` — `1 − max Tanimoto to training set`, with a heavy `-5`
  penalty for exact duplicates (drives diversity)

Combined: `fitness = 0.5·rf + 0.3·dissim + size_penalty`.

`tanimoto_kernel` is provided by the VQGAE package — you don't need to
re-implement it per notebook.


In [ ]:
def fitness_func_batch(_ga, solutions, _solutions_indices):
    counts = np.array(solutions)
    rf = rf_model.predict_proba(counts)[:, 1]
    size_penalty = np.where(counts.sum(-1) < 18, -1.0, 0.0)
    dissim = 1 - tanimoto_kernel(counts, X).max(-1)
    dissim += np.where(dissim == 0, -5, 0)
    return (0.5 * rf + 0.3 * dissim + size_penalty).tolist()

## Run the genetic algorithm

In [ ]:
NUM_GENS = 30   # 30 in the published runs; drop to 5 for a quick smoke run
initial_pop = X
num_parents = int(initial_pop.shape[0] * 0.33 // 10 * 10)   # ~190
keep_parents = int(num_parents * 0.66 // 10 * 10)           # ~120
print("parents mating:", num_parents, "kept:", keep_parents)

pbar = tqdm(total=NUM_GENS)
def _on_gen(_ga): pbar.update(1)

ga = pygad.GA(
    fitness_func=fitness_func_batch,
    on_generation=_on_gen,
    initial_population=initial_pop,
    num_genes=initial_pop.shape[-1],
    fitness_batch_size=BATCH,
    num_generations=NUM_GENS,
    num_parents_mating=num_parents,
    parent_selection_type="rws",
    crossover_type="single_point",
    mutation_type="adaptive",
    mutation_percent_genes=[10, 5],
    save_best_solutions=False,
    save_solutions=True,
    keep_elitism=0,
    keep_parents=keep_parents,
    suppress_warnings=True,
    random_seed=42,
    gene_type=int,
)
ga.run()
pbar.close()
ga.plot_fitness()

## Dedupe and rescore the GA's accumulated solutions

In [ ]:
solutions = list({tuple(s) for s in ga.solutions})
print(f"{len(solutions)} unique candidates")

scores = {"rf_score": [], "similarity_score": []}
for i in tqdm(range(0, len(solutions), 100), desc="rescore"):
    chunk = np.array(solutions[i:i + 100])
    scores["rf_score"].extend(rf_model.predict_proba(chunk)[:, 1].tolist())
    scores["similarity_score"].extend(tanimoto_kernel(chunk, X).max(-1).tolist())
sc_df = pd.DataFrame(scores)
sc_df.describe()

## Pick promising candidates and decode them

In [ ]:
chosen = sc_df[(sc_df["similarity_score"] < 0.95) & (sc_df["rf_score"] > 0.5)]
print(f"{len(chosen)} candidates pass coarse filters")

ids = chosen.index.to_list()
chosen_solutions = np.array([solutions[i] for i in ids])

vqgae_dec = VQGAE.load_from_checkpoint(vqgae_ckpt, task="decode", batch_size=100, map_location="cpu").eval()

# decode_population wraps frag_counts_to_inds → restore_order → decode_molecules
mols, validity, ordering_scores = decode_population(
    chosen_solutions, vqgae_dec, ordering_model, batch_size=100
)
print(f"{sum(validity)}/{len(validity)} decode to valid molecules")

## Structure filters

`filter_molecule` (provided by VQGAE.utils) drops common bad groups
(allene, peroxide, peroxide_charge, certain hetero triplets, fused
weird rings) and ring sizes outside [4, 8]. Apply it after the
`validity` flag from `decode_population`.


In [ ]:
final_mols, final_records = [], []
for mol, ok, rf_sc, sim_sc, ord_sc, idx in zip(
    mols, validity, chosen.rf_score, chosen.similarity_score, ordering_scores, chosen.index
):
    if not ok or not filter_molecule(mol):
        continue
    mol.meta.update({"rf_score": rf_sc, "similarity_score": sim_sc, "ordering_score": ord_sc})
    final_mols.append(mol)
    final_records.append({"smiles": str(mol), "rf_score": rf_sc,
                          "similarity_score": sim_sc, "ordering_score": ord_sc})
print(f"{len(final_mols)} molecules survive structure filters")
pd.DataFrame(final_records).head()

## Save

In [ ]:
out_path = "tubulin_chosen_generated.sdf"
with SDFWrite(out_path) as out:
    for mol in final_mols:
        out.write(mol)
print("wrote", out_path)

---

## How the data fixture was produced

The two files in `data/tubulin/` were built once from `tubulin_train_val.sdf`
(603 actives + inactives with `meta["activity"]` labels):

```python
from VQGAE import vqgae_encode_dataset, frag_inds_to_counts
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, GridSearchCV

# encode the SDF in batches
codebook_inds = vqgae_encode_dataset("tubulin_train_val.sdf", encoder, batch_size=10)
X = frag_inds_to_counts(codebook_inds).astype(np.int64)

# fit a 10-fold CV RF
kf = KFold(n_splits=10, shuffle=True, random_state=42)
grid = {"n_estimators": list(range(100, 550, 50)), "max_features": ["sqrt", "log2"]}
rf_cls = GridSearchCV(RandomForestClassifier(), grid, scoring="balanced_accuracy", n_jobs=10, cv=kf)
rf_cls.fit(X, Y)
```
